In [ ]:
#!pip install pandas numpy matplotlib seaborn scikit-learn statsmodels openpyxl

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [2]:
# Load the data
excel_file = 'SOBSS Branded PS Data 2023.xlsx'

# Inspect available sheets
xls = pd.ExcelFile(excel_file)
print("Available sheets:")
print(xls.sheet_names)
print("\n")

# Read all data sheets (exclude the Data Dictionary)
data_sheets = [s for s in xls.sheet_names if s.lower().strip() != 'data dictionary']
print(f"Loading sheets: {data_sheets}")

df_list = []
for s in data_sheets:
    df_tmp = pd.read_excel(excel_file, sheet_name=s)
    df_tmp['source_sheet'] = s
    df_list.append(df_tmp)

# Concatenate all region/channel sheets
df_raw = pd.concat(df_list, ignore_index=True)


print(f"Combined raw shape: {df_raw.shape}")



Available sheets:
['Data Dictionary', 'US SEO', 'US PPC', 'Canada PPC', 'Canada SEO']


Loading sheets: ['US SEO', 'US PPC', 'Canada PPC', 'Canada SEO']
Combined raw shape: (13365, 14)


In [3]:
# Convert day of month to day of week (categorical: Monday, Tuesday, etc.)
# Create a temporary datetime column to extract day of week
df_raw['temp_date'] = pd.to_datetime(df_raw[['year', 'month', 'day']].astype(str).agg('-'.join, axis=1))
df_raw['day_of_week'] = df_raw['temp_date'].dt.day_name()

# Convert to categorical for better analysis
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_raw['day_of_week'] = pd.Categorical(df_raw['day_of_week'], categories=day_order, ordered=True)

# Drop the temporary date column
df_raw = df_raw.drop(columns=['temp_date'])

print('\nDay of week conversion complete:')
print(f"Day of week categories: {df_raw['day_of_week'].cat.categories.tolist()}")
print(f"Unique day_of_week values: {sorted(df_raw['day_of_week'].unique())}")




Day of week conversion complete:
Day of week categories: ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
Unique day_of_week values: ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']


In [4]:
# Create dummy variables for day of week (6 dummies, Sunday as reference)
day_dummies = pd.get_dummies(df_raw['day_of_week'], prefix='day', drop_first=True)
month_dummies = pd.get_dummies(df_raw['month'], prefix='month', drop_first=True).astype(int)
hour_dummies = pd.get_dummies(df_raw['Hour'], prefix='hour', drop_first=True).astype(int)


# Convert boolean dummies to int for regression
day_dummies = day_dummies.astype(int)

# Add dummy columns to df_raw
df_raw = pd.concat([df_raw, month_dummies, hour_dummies, day_dummies], axis=1)

month_cols = [col for col in df_raw.columns if col.startswith('month_')]
hour_cols = [col for col in df_raw.columns if col.startswith('hour_')]
day_cols = [col for col in df_raw.columns if col.startswith('day_')]

control_vars = ' + '.join(month_cols + hour_cols + day_cols)

print('\nDay of week dummy variables created:')
print(f"Dummy columns: {day_dummies.columns.tolist()}")
print(f"Reference category: Sunday (omitted to avoid multicollinearity)")
print(f"Shape after adding dummies: {df_raw.shape}")




Day of week dummy variables created:
Dummy columns: ['day_Tuesday', 'day_Wednesday', 'day_Thursday', 'day_Friday', 'day_Saturday', 'day_Sunday']
Reference category: Sunday (omitted to avoid multicollinearity)
Shape after adding dummies: (13365, 48)


In [5]:
# Normalize some column names for easier handling
rename_map = {
    'BRAND_ads_ON_Flag (valid during test period only)': 'BRAND_ads_ON_Flag',
    'Hour Designation (during test period)': 'Hour_Designation',
    'Test Preiod (1 = Test Period, 0 = Non-test period)': 'Test_Period'
}

df_raw = df_raw.rename(columns=rename_map)

df_raw = df_raw.drop(columns= ['Hour_Designation','cnt_obsrv'])

print('\nColumns after rename:')
print(df_raw.columns.tolist())




Columns after rename:
['Country', 'Channel', 'year', 'month', 'day', 'Hour', 'Test_Period', 'BRAND_ads_ON_Flag', 'Users', 'NewUsers', 'Trials', 'source_sheet', 'day_of_week', 'month_6', 'month_7', 'month_8', 'month_9', 'hour_1', 'hour_2', 'hour_3', 'hour_4', 'hour_5', 'hour_6', 'hour_7', 'hour_8', 'hour_9', 'hour_10', 'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16', 'hour_17', 'hour_18', 'hour_19', 'hour_20', 'hour_21', 'hour_22', 'hour_23', 'day_Tuesday', 'day_Wednesday', 'day_Thursday', 'day_Friday', 'day_Saturday', 'day_Sunday']


In [18]:
# Aggregation keys (per country totals) - now using day of week instead of day of month
agg_keys = ['year', 'month', 'day_of_week', 'Hour', 'BRAND_ads_ON_Flag']
for k in agg_keys:
    if k not in df_raw.columns:
        raise KeyError(f"Missing aggregation key: {k}")

# Filter to only test period
df_test = df_raw[df_raw['Test_Period'] == 1]

# Total for United States
df_total_US = (
    df_test[df_test['Country'].str.lower().str.contains('united')]
    .groupby(agg_keys)[['Users', 'NewUsers', 'Trials', 'day_Tuesday', 'day_Wednesday', 'day_Thursday', 'day_Friday', 'day_Saturday', 'day_Sunday']]
    .sum()
    .reset_index()
    .rename(columns={'Users': 'total_users_US', 'NewUsers': 'total_newusers_US', 'Trials': 'total_trials_US'})
)

# Total for Canada
df_total_Canada = (
    df_test[df_test['Country'].str.lower().str.contains('canada')]
    .groupby(agg_keys)[['Users', 'NewUsers', 'Trials', 'day_Tuesday', 'day_Wednesday', 'day_Thursday', 'day_Friday', 'day_Saturday', 'day_Sunday']]
    .sum()
    .reset_index()
    .rename(columns={'Users': 'total_users_CA', 'NewUsers': 'total_newusers_CA', 'Trials': 'total_trials_CA'})
)

print(f"\nTotal US aggregated shape: {df_total_US.shape}")
print(f"Total Canada aggregated shape: {df_total_Canada.shape}")

# Show a few rows of each aggregated DF
print('\nSample df_total_US:')
print(df_total_US.head())
print('\nSample df_total_Canada:')
print(df_total_Canada.head())


Total US aggregated shape: (336, 14)
Total Canada aggregated shape: (336, 14)

Sample df_total_US:
   year  month day_of_week  Hour  BRAND_ads_ON_Flag  total_users_US  \
0  2023      7      Monday     0                  0              43   
1  2023      7      Monday     0                  1               0   
2  2023      7      Monday     1                  0              24   
3  2023      7      Monday     1                  1               0   
4  2023      7      Monday     2                  0               0   

   total_newusers_US  total_trials_US  day_Tuesday  day_Wednesday  \
0                 14                1            0              0   
1                  0                0            0              0   
2                 12                0            0              0   
3                  0                0            0              0   
4                  0                0            0              0   

   day_Thursday  day_Friday  day_Saturday  day_Sunday  
0 

In [19]:
# Filter out rows where BRAND_ads_ON_Flag = 1 but all outcome variables are 0
# These represent treatment periods with no data

print("Before filtering:")
print(f"US total shape: {df_total_US.shape}")
print(f"Canada total shape: {df_total_Canada.shape}")

# For US total
df_total_US = df_total_US[~(
    (df_total_US['BRAND_ads_ON_Flag'] == 1) & 
    (df_total_US['total_users_US'] == 0) & 
    (df_total_US['total_newusers_US'] == 0) & 
    (df_total_US['total_trials_US'] == 0)
)]

# For Canada total
df_total_Canada = df_total_Canada[~(
    (df_total_Canada['BRAND_ads_ON_Flag'] == 1) & 
    (df_total_Canada['total_users_CA'] == 0) & 
    (df_total_Canada['total_newusers_CA'] == 0) & 
    (df_total_Canada['total_trials_CA'] == 0)
)]

print("After filtering:")
print(f"US total shape: {df_total_US.shape}")
print(f"Canada total shape: {df_total_Canada.shape}")

Before filtering:
US total shape: (336, 14)
Canada total shape: (336, 14)
After filtering:
US total shape: (252, 14)
Canada total shape: (251, 14)


In [8]:
df_total_Canada.head()

,year,month,day_of_week,Hour,BRAND_ads_ON_Flag,total_users_CA,total_newusers_CA,total_trials_CA,day_Tuesday,day_Wednesday,day_Thursday,day_Friday,day_Saturday,day_Sunday
0,2023,5,Monday,0,0,57,22,3,0,0,0,0,0,0
2,2023,5,Monday,1,0,37,9,4,0,0,0,0,0,0
4,2023,5,Monday,2,0,0,0,0,0,0,0,0,0,0
5,2023,5,Monday,2,1,18,6,2,0,0,0,0,0,0
6,2023,5,Monday,3,0,0,0,0,0,0,0,0,0,0


In [9]:
df_total_US.head()

,year,month,day_of_week,Hour,BRAND_ads_ON_Flag,total_users_US,total_newusers_US,total_trials_US,day_Tuesday,day_Wednesday,day_Thursday,day_Friday,day_Saturday,day_Sunday
0,2023,5,Monday,0,0,106,42,0,0,0,0,0,0,0
2,2023,5,Monday,1,0,92,25,2,0,0,0,0,0,0
4,2023,5,Monday,2,0,0,0,0,0,0,0,0,0,0
5,2023,5,Monday,2,1,54,20,1,0,0,0,0,0,0
6,2023,5,Monday,3,0,0,0,0,0,0,0,0,0,0


In [10]:
# Create 4 raw dataframes for each country and channel combination
# US SEO
df_us_seo = df_raw[(df_raw['Country'].str.lower().str.contains('united')) & (df_raw['source_sheet'] == 'US SEO')].copy()

# US PPC
df_us_ppc = df_raw[(df_raw['Country'].str.lower().str.contains('united')) & (df_raw['source_sheet'] == 'US PPC')].copy()

# Canada SEO
df_ca_seo = df_raw[(df_raw['Country'].str.lower().str.contains('canada')) & (df_raw['source_sheet'] == 'Canada SEO')].copy()

# Canada PPC
df_ca_ppc = df_raw[(df_raw['Country'].str.lower().str.contains('canada')) & (df_raw['source_sheet'] == 'Canada PPC')].copy()

print(f"US SEO shape: {df_us_seo.shape}")
print(f"US PPC shape: {df_us_ppc.shape}")
print(f"Canada SEO shape: {df_ca_seo.shape}")
print(f"Canada PPC shape: {df_ca_ppc.shape}")


US SEO shape: (3453, 46)
US PPC shape: (3410, 46)
Canada SEO shape: (3345, 46)
Canada PPC shape: (3157, 46)


In [11]:
df_us_ppc.head()

,Country,Channel,year,month,day,Hour,Test_Period,BRAND_ads_ON_Flag,Users,NewUsers,...,hour_20,hour_21,hour_22,hour_23,day_Tuesday,day_Wednesday,day_Thursday,day_Friday,day_Saturday,day_Sunday
3453,United States,PPC Branded,2023,5,1,0,0,0,9,4,...,0,0,0,0,0,0,0,0,0,0
3454,United States,PPC Branded,2023,5,1,1,0,0,8,4,...,0,0,0,0,0,0,0,0,0,0
3455,United States,PPC Branded,2023,5,1,2,0,1,8,7,...,0,0,0,0,0,0,0,0,0,0
3456,United States,PPC Branded,2023,5,1,3,0,1,5,3,...,0,0,0,0,0,0,0,0,0,0
3457,United States,PPC Branded,2023,5,1,4,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
# Create dummy variables for month, hour, and day_of_week in the aggregated dataframes

# For US data
month_dummies_US = pd.get_dummies(df_total_US['month'], prefix='month', drop_first=True).astype(int)
hour_dummies_US = pd.get_dummies(df_total_US['Hour'], prefix='hour', drop_first=True).astype(int)
day_dummies_US = pd.get_dummies(df_total_US['day_of_week'], prefix='day', drop_first=True).astype(int)

df_total_US = pd.concat([df_total_US, month_dummies_US, hour_dummies_US, day_dummies_US], axis=1)
df_total_US = df_total_US.drop(columns=['month', 'Hour', 'day_of_week'])

# For Canada data
month_dummies_CA = pd.get_dummies(df_total_Canada['month'], prefix='month', drop_first=True).astype(int)
hour_dummies_CA = pd.get_dummies(df_total_Canada['Hour'], prefix='hour', drop_first=True).astype(int)
day_dummies_CA = pd.get_dummies(df_total_Canada['day_of_week'], prefix='day', drop_first=True).astype(int)

df_total_Canada = pd.concat([df_total_Canada, month_dummies_CA, hour_dummies_CA, day_dummies_CA], axis=1)
df_total_Canada = df_total_Canada.drop(columns=['month', 'Hour', 'day_of_week'])

print("Dummies created for US and Canada data, original categorical columns dropped.")

Dummies created for US and Canada data, original categorical columns dropped.


In [21]:
# Run 6 different difference-in-difference regressions
# Assuming DiD with treatment = BRAND_ads_ON_Flag, response = trials (and other outcomes), controls = day, month, hour dummies

import statsmodels.formula.api as smf

# Get the list of dummy columns from the aggregated data
month_cols = [col for col in df_total_US.columns if col.startswith('month_')]
hour_cols = [col for col in df_total_US.columns if col.startswith('hour_')]
day_cols = [col for col in df_total_US.columns if col.startswith('day_')]
control_vars = ' + '.join(month_cols + hour_cols + day_cols)

# Regression 1: US Trials
formula1 = f'total_trials_US ~ BRAND_ads_ON_Flag + {control_vars}'
model1 = smf.ols(formula1, data=df_total_US).fit()
print("Regression 1: US Trials")
model1.summary()


Regression 1: US Trials


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:        total_trials_US   R-squared:                       0.666
Model:                            OLS   Adj. R-squared:                  0.610
Method:                 Least Squares   F-statistic:                     11.89
Date:                Fri, 09 Jan 2026   Prob (F-statistic):           2.85e-34
Time:                        14:25:01   Log-Likelihood:                -550.44
No. Observations:                 252   AIC:                             1175.
Df Residuals:                     215   BIC:                             1305.
Df Model:                          36                                         
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -0.1389      0.986     -0.141      0.888      -2.083       1.805
BRAND_ads_ON_Flag     3.1759      0.836      3.798      0.000       1.528       4.824
hour_1                0.3454      1.245      0.278      0.782      -2.108       2.799
hour_2               -1.5235      1.329     -1.147      0.253      -4.142       1.095
hour_3               -2.1236      1.338     -1.587      0.114      -4.761       0.513
hour_4               -1.0776      1.256     -0.858      0.392      -3.553       1.398
hour_5               -1.0437      1.258     -0.830      0.408      -3.522       1.435
hour_6               -1.8092      1.329     -1.362      0.175      -4.428       0.810
hour_7               -1.6664      1.329     -1.254      0.211      -4.285       0.953
hour_8                0.4286      1.244      0.344      0.731      -2.024       2.881
hour_9                2.5714      1.244      2.067      0.040       0.119       5.024
hour_10               1.7622      1.329      1.326      0.186      -0.857       4.381
hour_11               2.1194      1.329      1.595      0.112      -0.500       4.738
hour_12               4.5714      1.244      3.675      0.000       2.119       7.024
hour_13               5.2857      1.244      4.249      0.000       2.834       7.738
hour_14               1.9051      1.329      1.434      0.153      -0.714       4.524
hour_15               2.8336      1.329      2.133      0.034       0.215       5.453
hour_16               5.2857      1.244      4.249      0.000       2.834       7.738
hour_17               5.1429      1.244      4.134      0.000       2.691       7.595
hour_18               1.2622      1.329      0.950      0.343      -1.357       3.881
hour_19               0.1194      1.329      0.090      0.929      -2.500       2.738
hour_20               1.7143      1.244      1.378      0.170      -0.738       4.166
hour_21               2.5714      1.244      2.067      0.040       0.119       5.024
hour_22              -0.3806      1.329     -0.286      0.775      -3.000       2.238
hour_23              -0.9521      1.329     -0.717      0.474      -3.571       1.667
day_Tuesday[0]        0.5991      0.294      2.038      0.043       0.020       1.178
day_Tuesday[1]       -0.0476      0.937     -0.051      0.960      -1.894       1.799
day_Wednesday[0]      0.5456      0.292      1.867      0.063      -0.030       1.121
day_Wednesday[1]     -0.0079      0.940     -0.008      0.993      -1.860       1.844
day_Thursday[0]       0.6794      0.292      2.325      0.021       0.103       1.255
day_Thursday[1]      -0.0240      0.940     -0.025      0.980      -1.876       1.828
day_Friday[0]         0.4180      0.291      1.434      0.153      -0.156       0.992
day_Friday[1]         0.0196      0.938      0.021      0.983      -1.829

In [22]:
# Regression 2: Canada Trials
formula2 = f'total_trials_CA ~ BRAND_ads_ON_Flag + {control_vars}'
model2 = smf.ols(formula2, data=df_total_Canada).fit()
print("Regression 2: Canada Trials")
model2.summary()

Regression 2: Canada Trials


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:        total_trials_CA   R-squared:                       0.404
Model:                            OLS   Adj. R-squared:                  0.304
Method:                 Least Squares   F-statistic:                     4.031
Date:                Fri, 09 Jan 2026   Prob (F-statistic):           8.05e-11
Time:                        14:25:05   Log-Likelihood:                -295.31
No. Observations:                 251   AIC:                             664.6
Df Residuals:                     214   BIC:                             795.1
Df Model:                          36                                         
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -0.0114      0.350     -0.033      0.974      -0.701       0.678
BRAND_ads_ON_Flag     0.3451      0.274      1.260      0.209      -0.195       0.885
hour_1               -0.1252      0.456     -0.275      0.784      -1.023       0.773
hour_2               -0.2206      0.445     -0.496      0.621      -1.098       0.657
hour_3               -0.0822      0.445     -0.185      0.854      -0.960       0.796
hour_4               -0.0474      0.459     -0.103      0.918      -0.953       0.858
hour_5               -0.1796      0.458     -0.392      0.696      -1.083       0.724
hour_6               -0.1382      0.452     -0.305      0.760      -1.030       0.753
hour_7               -0.2517      0.440     -0.572      0.568      -1.119       0.616
hour_8                0.8532      0.458      1.864      0.064      -0.049       1.755
hour_9                0.6910      0.460      1.502      0.135      -0.216       1.598
hour_10               0.3059      0.438      0.699      0.486      -0.557       1.169
hour_11               0.4456      0.436      1.021      0.308      -0.415       1.306
hour_12               0.6910      0.460      1.502      0.135      -0.216       1.598
hour_13               0.5482      0.460      1.191      0.235      -0.359       1.455
hour_14               0.5170      0.436      1.185      0.238      -0.343       1.377
hour_15               0.7313      0.436      1.675      0.095      -0.129       1.592
hour_16               0.9767      0.460      2.123      0.035       0.070       1.884
hour_17               0.2624      0.460      0.570      0.569      -0.645       1.169
hour_18               0.1598      0.436      0.366      0.715      -0.700       1.020
hour_19              -0.0544      0.436     -0.125      0.901      -0.915       0.806
hour_20               1.1325      0.458      2.471      0.014       0.229       2.036
hour_21               0.7820      0.457      1.712      0.088      -0.118       1.682
hour_22              -0.0512      0.438     -0.117      0.907      -0.914       0.812
hour_23               0.0380      0.439      0.086      0.931      -0.828       0.904
day_Tuesday[0]        0.1516      0.103      1.468      0.144      -0.052       0.355
day_Tuesday[1]       -0.1026      0.311     -0.330      0.742      -0.715       0.510
day_Wednesday[0]      0.1930      0.104      1.853      0.065      -0.012       0.398
day_Wednesday[1]     -0.1532      0.317     -0.484      0.629      -0.777       0.471
day_Thursday[0]       0.1152      0.103      1.118      0.265      -0.088       0.318
day_Thursday[1]      -0.0839      0.317     -0.265      0.791      -0.709       0.541
day_Friday[0]         0.0941      0.101      0.933      0.352      -0.105       0.293
day_Friday[1]        -0.0465      0.324     -0.144      0.886      -0.685

In [24]:
# # Regression 3: US Users
# formula3 = f'total_users_US ~ BRAND_ads_ON_Flag + {control_vars}'
# model3 = smf.ols(formula3, data=df_total_US).fit()
# print("Regression 3: US Users")
# print(model3.summary())
# print("\n" + "="*50 + "\n")

# # Regression 4: Canada Users
# formula4 = f'total_users_CA ~ BRAND_ads_ON_Flag + {control_vars}'
# model4 = smf.ols(formula4, data=df_total_Canada).fit()
# print("Regression 4: Canada Users")
# print(model4.summary())
# print("\n" + "="*50 + "\n")

# # Regression 5: US New Users
# formula5 = f'total_newusers_US ~ BRAND_ads_ON_Flag + {control_vars}'
# model5 = smf.ols(formula5, data=df_total_US).fit()
# print("Regression 5: US New Users")
# print(model5.summary())
# print("\n" + "="*50 + "\n")

# # Regression 6: Canada New Users
# formula6 = f'total_newusers_CA ~ BRAND_ads_ON_Flag + {control_vars}'
# model6 = smf.ols(formula6, data=df_total_Canada).fit()
# print("Regression 6: Canada New Users")
# print(model6.summary())

In [25]:
# Filter to test period for each channel
df_us_seo_test = df_us_seo[df_us_seo['Test_Period'] == 1]
df_us_ppc_test = df_us_ppc[df_us_ppc['Test_Period'] == 1]
df_ca_seo_test = df_ca_seo[df_ca_seo['Test_Period'] == 1]
df_ca_ppc_test = df_ca_ppc[df_ca_ppc['Test_Period'] == 1]

print(f"US SEO test shape: {df_us_seo_test.shape}")
print(f"US PPC test shape: {df_us_ppc_test.shape}")
print(f"Canada SEO test shape: {df_ca_seo_test.shape}")
print(f"Canada PPC test shape: {df_ca_ppc_test.shape}")

US SEO test shape: (384, 46)
US PPC test shape: (363, 46)
Canada SEO test shape: (365, 46)
Canada PPC test shape: (305, 46)


In [26]:
# Regressions for US SEO
print("Regressions for US SEO")
print("="*50)

# Regression 1: US SEO Trials
formula1 = f'Trials ~ BRAND_ads_ON_Flag + {control_vars}'
model1 = smf.ols(formula1, data=df_us_seo_test).fit()
print("Regression 1: US SEO Trials")
print(model1.summary())
print("\n" + "="*50 + "\n")

# # Regression 2: US SEO Users
# formula2 = f'Users ~ BRAND_ads_ON_Flag + {control_vars}'
# model2 = smf.ols(formula2, data=df_us_seo_test).fit()
# print("Regression 2: US SEO Users")
# print(model2.summary())
# print("\n" + "="*50 + "\n")

# # Regression 3: US SEO New Users
# formula3 = f'NewUsers ~ BRAND_ads_ON_Flag + {control_vars}'
# model3 = smf.ols(formula3, data=df_us_seo_test).fit()
# print("Regression 3: US SEO New Users")
# print(model3.summary())
# print("\n" + "="*50 + "\n")

Regressions for US SEO
Regression 1: US SEO Trials
                            OLS Regression Results                            
Dep. Variable:                 Trials   R-squared:                       0.413
Model:                            OLS   Adj. R-squared:                  0.365
Method:                 Least Squares   F-statistic:                     8.598
Date:                Fri, 09 Jan 2026   Prob (F-statistic):           1.26e-26
Time:                        14:44:11   Log-Likelihood:                -597.87
No. Observations:                 384   AIC:                             1256.
Df Residuals:                     354   BIC:                             1374.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------

In [27]:
# Regressions for US PPC
print("Regressions for US PPC")
print("="*50)

# Regression 1: US PPC Trials
formula1 = f'Trials ~ BRAND_ads_ON_Flag + {control_vars}'
model1 = smf.ols(formula1, data=df_us_ppc_test).fit()
print("Regression 1: US PPC Trials")
print(model1.summary())
print("\n" + "="*50 + "\n")

# # Regression 2: US PPC Users
# formula2 = f'Users ~ BRAND_ads_ON_Flag + {control_vars}'
# model2 = smf.ols(formula2, data=df_us_ppc_test).fit()
# print("Regression 2: US PPC Users")
# print(model2.summary())
# print("\n" + "="*50 + "\n")

# # Regression 3: US PPC New Users
# formula3 = f'NewUsers ~ BRAND_ads_ON_Flag + {control_vars}'
# model3 = smf.ols(formula3, data=df_us_ppc_test).fit()
# print("Regression 3: US PPC New Users")
# print(model3.summary())
# print("\n" + "="*50 + "\n")

Regressions for US PPC
Regression 1: US PPC Trials
                            OLS Regression Results                            
Dep. Variable:                 Trials   R-squared:                       0.441
Model:                            OLS   Adj. R-squared:                  0.393
Method:                 Least Squares   F-statistic:                     9.068
Date:                Fri, 09 Jan 2026   Prob (F-statistic):           1.09e-27
Time:                        14:44:20   Log-Likelihood:                -433.54
No. Observations:                 363   AIC:                             927.1
Df Residuals:                     333   BIC:                             1044.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------

In [28]:
# Regressions for Canada SEO
print("Regressions for Canada SEO")
print("="*50)

# Regression 1: Canada SEO Trials
formula1 = f'Trials ~ BRAND_ads_ON_Flag + {control_vars}'
model1 = smf.ols(formula1, data=df_ca_seo_test).fit()
print("Regression 1: Canada SEO Trials")
print(model1.summary())
print("\n" + "="*50 + "\n")

# # Regression 2: Canada SEO Users
# formula2 = f'Users ~ BRAND_ads_ON_Flag + {control_vars}'
# model2 = smf.ols(formula2, data=df_ca_seo_test).fit()
# print("Regression 2: Canada SEO Users")
# print(model2.summary())
# print("\n" + "="*50 + "\n")

# # Regression 3: Canada SEO New Users
# formula3 = f'NewUsers ~ BRAND_ads_ON_Flag + {control_vars}'
# model3 = smf.ols(formula3, data=df_ca_seo_test).fit()
# print("Regression 3: Canada SEO New Users")
# print(model3.summary())
# print("\n" + "="*50 + "\n")

Regressions for Canada SEO
Regression 1: Canada SEO Trials
                            OLS Regression Results                            
Dep. Variable:                 Trials   R-squared:                       0.157
Model:                            OLS   Adj. R-squared:                  0.084
Method:                 Least Squares   F-statistic:                     2.156
Date:                Fri, 09 Jan 2026   Prob (F-statistic):           0.000695
Time:                        14:44:24   Log-Likelihood:                -274.38
No. Observations:                 365   AIC:                             608.8
Df Residuals:                     335   BIC:                             725.8
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------

In [29]:
# Regressions for Canada PPC
print("Regressions for Canada PPC")
print("="*50)

# Regression 1: Canada PPC Trials
formula1 = f'Trials ~ BRAND_ads_ON_Flag + {control_vars}'
model1 = smf.ols(formula1, data=df_ca_ppc_test).fit()
print("Regression 1: Canada PPC Trials")
print(model1.summary())
print("\n" + "="*50 + "\n")

# # Regression 2: Canada PPC Users
# formula2 = f'Users ~ BRAND_ads_ON_Flag + {control_vars}'
# model2 = smf.ols(formula2, data=df_ca_ppc_test).fit()
# print("Regression 2: Canada PPC Users")
# print(model2.summary())
# print("\n" + "="*50 + "\n")

# # Regression 3: Canada PPC New Users
# formula3 = f'NewUsers ~ BRAND_ads_ON_Flag + {control_vars}'
# model3 = smf.ols(formula3, data=df_ca_ppc_test).fit()
# print("Regression 3: Canada PPC New Users")
# print(model3.summary())
# print("\n" + "="*50 + "\n")

Regressions for Canada PPC
Regression 1: Canada PPC Trials
                            OLS Regression Results                            
Dep. Variable:                 Trials   R-squared:                       0.154
Model:                            OLS   Adj. R-squared:                  0.065
Method:                 Least Squares   F-statistic:                     1.730
Date:                Fri, 09 Jan 2026   Prob (F-statistic):             0.0137
Time:                        14:44:29   Log-Likelihood:                -96.472
No. Observations:                 305   AIC:                             252.9
Df Residuals:                     275   BIC:                             364.6
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------